In [1]:
import sys
import pandas as pd
import numpy as np
import sqlite3

In [2]:
raw = pd.read_excel("/Users/linarojas/Desktop/Research/Papers/TernOpt/data/Ternary_Optical_data.xlsx", sheet_name='dielectric')
df_compositions = pd.read_excel("/Users/linarojas/Desktop/Research/Papers/TernOpt/data/Ternary_Optical_data.xlsx", sheet_name='Composition')

In [5]:
import pandas as pd

wavelength = raw.iloc[:, 0].to_numpy()
blocks = raw.iloc[:, 1:]

# Sanity check: everything after Wavelength should come in groups of 3
assert blocks.shape[1] % 3 == 0, f"{blocks.shape[1]} columns isn't a multiple of 3"

e1_rows, e2_rows = {}, {}

for i in range(0, blocks.shape[1], 3):
    id_col, e1_col, e2_col = blocks.columns[i:i + 3]

    # Use the value in the ID column if there is one; otherwise use the header
    ids_in_col = blocks[id_col].dropna()
    sample_id = str(ids_in_col.iloc[0]) if len(ids_in_col) else str(id_col)

    if sample_id in e1_rows:
        print(f"Warning: duplicate ID {sample_id}")

    e1_rows[sample_id] = blocks[e1_col].to_numpy()
    e2_rows[sample_id] = blocks[e2_col].to_numpy()

e1 = pd.DataFrame.from_dict(e1_rows, orient="index", columns=wavelength)
e2 = pd.DataFrame.from_dict(e2_rows, orient="index", columns=wavelength)
e1.index.name = e2.index.name = "ID"
e1.columns.name = e2.columns.name = "wavelength"

print(e1.shape)   # (number of IDs, number of wavelengths)
e1.head()

(141, 1041)


wavelength,210,211,212,213,214,215,216,217,218,219,...,2446,2452,2458,2464,2470,2476,2482,2488,2494,2500
ID,,,,,,,,,,,,,,,,,,,,,
1,-1.133783,-1.260955,-1.380693,-1.488621,-1.581998,-1.659146,-1.719189,-1.761921,-1.787732,-1.797581,...,-274.689667,-275.918243,-277.147369,-278.377045,-279.607208,-280.837860,-282.068939,-283.300446,-284.532288,-285.764465
2,-1.857539,-2.017619,-2.170887,-2.311307,-2.435062,-2.539764,-2.624113,-2.687706,-2.730934,-2.754954,...,-75.781296,-75.983788,-76.185783,-76.387260,-76.588188,-76.788559,-76.988342,-77.187523,-77.386086,-77.584007
3,-2.701878,-2.715684,-2.736772,-2.763758,-2.795585,-2.831381,-2.870400,-2.911993,-2.955572,-3.000602,...,-324.157654,-325.182312,-326.201996,-327.216614,-328.226044,-329.230194,-330.228973,-331.222260,-332.209961,-333.191956
4,-1.388275,-1.518159,-1.640196,-1.749701,-1.843735,-1.920488,-1.979004,-2.019035,-2.040963,-2.045769,...,-202.311310,-203.053146,-203.792709,-204.529968,-205.264847,-205.997299,-206.727264,-207.454666,-208.179459,-208.901535
5,-1.264434,-1.491116,-1.703400,-1.893279,-2.055785,-2.187916,-2.288167,-2.356279,-2.393100,-2.400543,...,-263.507782,-264.584076,-265.658844,-266.732056,-267.803650,-268.873535,-269.941681,-271.008057,-272.072571,-273.135162


In [29]:
DB_PATH = "ternopt.db"

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

In [30]:
comp = pd.read_sql("SELECT * FROM composition", conn, index_col="id")
e1 = pd.read_sql("SELECT * FROM e1", conn, index_col="id")
e1.columns = e1.columns.astype(int)

In [32]:
e1

,210,211,212,213,214,215,216,217,218,219,...,2446,2452,2458,2464,2470,2476,2482,2488,2494,2500
id,,,,,,,,,,,,,,,,,,,,,
1,-1.133783,-1.260955,-1.380693,-1.488621,-1.581998,-1.659146,-1.719189,-1.761921,-1.787732,-1.797581,...,-274.689667,-275.918243,-277.147369,-278.377045,-279.607208,-280.837860,-282.068939,-283.300446,-284.532288,-285.764465
2,-1.857539,-2.017619,-2.170887,-2.311307,-2.435062,-2.539764,-2.624113,-2.687706,-2.730934,-2.754954,...,-75.781296,-75.983788,-76.185783,-76.387260,-76.588188,-76.788559,-76.988342,-77.187523,-77.386086,-77.584007
3,-2.701878,-2.715684,-2.736772,-2.763758,-2.795585,-2.831381,-2.870400,-2.911993,-2.955572,-3.000602,...,-324.157654,-325.182312,-326.201996,-327.216614,-328.226044,-329.230194,-330.228973,-331.222260,-332.209961,-333.191956
4,-1.388275,-1.518159,-1.640196,-1.749701,-1.843735,-1.920488,-1.979004,-2.019035,-2.040963,-2.045769,...,-202.311310,-203.053146,-203.792709,-204.529968,-205.264847,-205.997299,-206.727264,-207.454666,-208.179459,-208.901535
5,-1.264434,-1.491116,-1.703400,-1.893279,-2.055785,-2.187916,-2.288167,-2.356279,-2.393100,-2.400543,...,-263.507782,-264.584076,-265.658844,-266.732056,-267.803650,-268.873535,-269.941681,-271.008057,-272.072571,-273.135162
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,-3.432870,-3.432230,-3.431581,-3.430913,-3.430212,-3.429468,-3.428670,-3.427808,-3.426872,-3.425853,...,-9.296460,-9.300356,-9.304233,-9.308089,-9.311928,-9.315747,-9.319549,-9.323331,-9.327095,-9.330841
138,-2.812268,-2.707620,-2.612192,-2.527074,-2.452819,-2.389616,-2.337368,-2.295725,-2.264110,-2.241710,...,-7.287261,-7.244594,-7.201409,-7.157696,-7.113439,-7.068625,-7.023246,-6.977288,-6.930742,-6.883592
139,-1.022624,-0.949110,-0.878531,-0.813414,-0.755355,-0.705350,-0.663934,-0.631268,-0.607174,-0.591155,...,-118.760765,-119.022789,-119.281982,-119.538284,-119.791664,-120.042053,-120.289421,-120.533714,-120.774879,-121.012856


In [33]:
conn.close()